In [0]:
# land raw FRED data
#
# fetches all configured FRED series (full history, latest vintage)
# each response lands as immutable JSON in the Catalog volume
# 
# fetches series-level metadata for all configured series and
# stores all as a single immutable JSON 
#
# stateless and idempotent

import json
import os
import time
import requests
from datetime import datetime, timezone


In [0]:
# configure 

CATALOG     = "fed_monitor"
VOLUME_ROOT = f"/Volumes/{CATALOG}/landing/fred"
 
FRED_OBS_URL  = "https://api.stlouisfed.org/fred/series/observations"
FRED_META_URL = "https://api.stlouisfed.org/fred/series"
 
API_KEY = dbutils.secrets.get(scope="fred", key="api_key")
 

In [0]:
# series list, group (e.g. "h41") is carried into landing envelope

SERIES = {
    # H.4.1 — Fed balance sheet (weekly Wednesday level, except RRPONTSYD = daily)
    "WALCL":          "h41",   # Total assets
    "WRESBAL":        "h41",   # Reserve balances
    "WTREGEN":        "h41",   # Treasury General Account
    "TREAST":         "h41",   # U.S. Treasuries held outright
    "WSHOMCB":        "h41",   # MBS held outright
    "RRPONTSYD":      "h41",   # ON RRP usage (daily)
    # H.8 — commercial bank balance sheets (weekly)
    "TOTBKCR":        "h8",    # Bank credit, all commercial banks
    "TOTCI":          "h8",    # C&I loans
    "DPSACBW027SBOG": "h8",    # Deposits
    "RELACBW027SBOG": "h8",    # Real estate loans
    "CLSACBW027SBOG": "h8",    # Consumer loans
    "CASACBW027SBOG": "h8",    # Cash assets
    "LNFACBW027SBOG": "h8",    # Loans to NDFIs
    "B2310NCBA": "h8",         # All other Loans not classified (IN MILLIONS)
    # Reference rates (daily)
    "SOFR":           "rates",
    "EFFR":           "rates",
    "IORB":           "rates",
    "DTB3":           "rates", # 3-month T-bill, secondary market
}


In [0]:
# fetch helpers


def fetch_with_retry(url: str, params: dict, max_retries: int = 3) -> dict:
    """GET with backoff on 429 / 5xx / network errors. Raises on 4xx
    (e.g. a bad series_id returns 400 — we want that surfaced, not retried)."""
    for attempt in range(max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
            if resp.status_code == 429 or resp.status_code >= 500:
                if attempt < max_retries:
                    base = 30.0 if resp.status_code == 429 else 1.0
                    time.sleep(min(base * (2 ** attempt), 60.0))
                    continue
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException:
            if attempt >= max_retries:
                raise
            time.sleep(min(2 ** attempt, 60.0))
 
 
def fetch_observations(series_id: str) -> dict:
    """Full history, latest vintages — raw payload returned untouched."""
    return fetch_with_retry(FRED_OBS_URL, {
        "series_id": series_id,
        "api_key":   API_KEY,
        "file_type": "json",
    })
 
 
def fetch_metadata(series_id: str) -> dict:
    """Series-level metadata (title, units, frequency, seasonal adjustment)."""
    return fetch_with_retry(FRED_META_URL, {
        "series_id": series_id,
        "api_key":   API_KEY,
        "file_type": "json",
    })
 

In [0]:
# landing writers

def write_json(path: str, payload: dict) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(payload, f)
 
 
def land_observations(series_id: str, group: str, run_ts: datetime) -> dict:
    payload = fetch_observations(series_id)
    n_obs = len(payload.get("observations", []))
 
    envelope = {
        "series_id":  series_id,
        "group":      group,
        "fetched_at": run_ts.isoformat(),
        "source_url": FRED_OBS_URL,
        "response":   payload,
    }
    path = (
        f"{VOLUME_ROOT}/observations"
        f"/series_id={series_id}"
        f"/fetch_date={run_ts:%Y-%m-%d}"
        f"/{series_id}_{run_ts:%Y%m%dT%H%M%SZ}.json"
    )
    write_json(path, envelope)
    return {"series_id": series_id, "n_obs": n_obs, "path": path}
 
 
def land_metadata(series_ids: list, run_ts: datetime) -> str:
    """One combined metadata file per run — the seed for dim_series."""
    records = []
    for sid in series_ids:
        meta = fetch_metadata(sid)
        for s in meta.get("seriess", []):
            records.append(s)
 
    envelope = {
        "fetched_at": run_ts.isoformat(),
        "source_url": FRED_META_URL,
        "series":     records,
    }
    path = (
        f"{VOLUME_ROOT}/series_metadata"
        f"/fetch_date={run_ts:%Y-%m-%d}"
        f"/series_metadata_{run_ts:%Y%m%dT%H%M%SZ}.json"
    )
    write_json(path, envelope)
    return path
 

In [0]:
# run

run_ts = datetime.now(timezone.utc)
results, failures = [], []
 
print(f"[{run_ts:%Y-%m-%d %H:%M:%S}Z] Landing {len(SERIES)} series → {VOLUME_ROOT}")
print("-" * 60)
 
for series_id, group in SERIES.items():
    try:
        r = land_observations(series_id, group, run_ts)
        results.append(r)
        print(f"  ok    {series_id:<16} {r['n_obs']:>6} obs")
    except Exception as e:
        failures.append((series_id, f"{type(e).__name__}: {str(e)[:120]}"))
        print(f"  FAIL  {series_id:<16} {failures[-1][1]}")
 
# Metadata pull (only for series that fetched successfully)
ok_ids = [r["series_id"] for r in results]
meta_path = None
if ok_ids:
    try:
        meta_path = land_metadata(ok_ids, run_ts)
        print(f"  ok    series_metadata ({len(ok_ids)} series)")
    except Exception as e:
        failures.append(("series_metadata", f"{type(e).__name__}: {str(e)[:120]}"))
        print(f"  FAIL  series_metadata  {failures[-1][1]}")
 

[2026-07-24 01:16:12Z] Landing 17 series → /Volumes/fed_monitor/landing/fred
------------------------------------------------------------
  ok    WALCL              1232 obs
  ok    WRESBAL            2221 obs
  ok    WTREGEN            2116 obs
  ok    TREAST             1232 obs
  ok    WSHOMCB            1232 obs
  ok    RRPONTSYD          6120 obs
  ok    TOTBKCR            2793 obs
  ok    TOTCI              2793 obs
  ok    DPSACBW027SBOG     2793 obs
  ok    RELACBW027SBOG     2793 obs
  ok    CLSACBW027SBOG     2793 obs
  ok    CASACBW027SBOG     2793 obs
  ok    LNFACBW027SBOG      601 obs
  ok    SOFR               2167 obs
  ok    EFFR               6798 obs
  ok    IORB               1822 obs
  ok    DTB3              18928 obs
  ok    series_metadata (17 series)


In [0]:
# summary prints

total_obs = sum(r["n_obs"] for r in results)
print("=" * 60)
print(f"Landed:   {len(results)}/{len(SERIES)} series, {total_obs:,} observations")
print(f"Metadata: {meta_path or 'not written'}")
if failures:
    print(f"Failures: {len(failures)}")
    for sid, err in failures:
        print(f"  - {sid}: {err}")
 
# Fail the (scheduled) run loudly if anything failed, after full reporting.
if failures:
    raise RuntimeError(
        f"{len(failures)} of {len(SERIES)} landing steps failed: "
        + ", ".join(sid for sid, _ in failures)
    )
 
print("All series landed successfully.")

Landed:   17/17 series, 61,227 observations
Metadata: /Volumes/fed_monitor/landing/fred/series_metadata/fetch_date=2026-07-24/series_metadata_20260724T011612Z.json
All series landed successfully.
